## Create HATS-sliver-of-DP2

The full DP2 HATS release occupies over 5TB of expensive storage on `data-int`. This notebook creates a subset of the data for testing. Ran on a gargantuan machine.

In [ ]:
import lsdb

from upath import UPath
from dask.distributed import Client

LSDB_DIR = UPath("/rubin/lsdb_data")
OUTPUT_DIR = LSDB_DIR / "sliver"

In [ ]:
from functools import reduce
from hats.pixel_math import region_to_moc


def _cones_to_moc(cones, max_depth=10):
    """Union a list of (ra, dec, radius_arcsec) cones into a single MOC."""
    mocs = (
        region_to_moc.cone_to_moc(
            ra=ra, dec=dec, radius_arcsec=radius_arcsec, max_depth=max_depth
        )
        for ra, dec, radius_arcsec in cones
    )
    return reduce(lambda a, b: a.union(b), mocs)


def main_moc():
    main_centers = {
        "DDF_ECDFS": (53.0, -28.1, 3600),  # 1deg
        "AT_2025wab": (53.311667, -29.588856, 360),  # 0.1deg
    }
    return _cones_to_moc(main_centers.values())


def photoz_moc():
    photoz_centers = {
        "DDF_ELAIS_S1": (9.5, -44.0),
        "DDF_ECDFS": (53.0, -28.1),
        "DDF_EDFS_b": (63.2, -47.8),
        "DDF_COSMOS": (150.1, 2.1),
        "New_Horizons": (289.4, -20.2),
        "Rubin_SV_225_-40": (225.0, -39.5),
        "Rubin_SV_320_-15": (320.2, -15.1),
    }
    radius = 2 * 3600
    cones = ((ra, dec, radius) for ra, dec in photoz_centers.values())
    return _cones_to_moc(cones)


def moc_for_collection(collection_name):
    match collection_name:
        case "dia_object_collection" | "object_collection":
            return main_moc()
        case "object_photoz":
            return photoz_moc()
    raise ValueError("Invalid catalog")

In [ ]:
from hats_import import pipeline_with_client
from hats_import.collection.arguments import CollectionArguments
from lsdb.core.search.region_search import MOCSearch
from hats.io.summary_file import (
    write_partition_info_png,
    write_skymap_png,
    write_catalog_summary_file,
)


def generate_collection(collection_name, main_cat_path, index_col):
    args = (
        CollectionArguments(
            output_artifact_name=collection_name,
            output_path=OUTPUT_DIR,
            simple_progress_bar=True,
            create_per_partition_stats=True,
            create_skymap_png=True,
            create_partition_info_png=True,
            create_summary_md=True,
            create_summary_html=True,
        )
        .catalog(catalog_path=main_cat_path)
        .add_margin(margin_threshold=5, is_default=True)
        .add_index(indexing_column=index_col)
    )
    pipeline_with_client(args, client)


def generate_sliver(collection_name):
    moc = moc_for_collection(collection_name)
    cat = lsdb.open_catalog(
        LSDB_DIR / collection_name, columns="all", search_filter=MOCSearch(moc)
    )
    main_cat_path = OUTPUT_DIR / collection_name / cat.name
    default_columns = cat.hc_structure.catalog_info.default_columns
    cat.write_catalog(
        main_cat_path,
        default_columns=default_columns,
        skymap_alt_orders=[2, 4, 6],
        as_collection=False,
    )
    write_skymap_png(main_cat_path)
    write_partition_info_png(main_cat_path)
    write_catalog_summary_file(main_cat_path, fmt="html")
    write_catalog_summary_file(main_cat_path, fmt="markdown")
    index_col = "objectId" if collection_name.startswith("object") else "diaObjectId"
    generate_collection(collection_name, main_cat_path, index_col)

In [ ]:
with Client(n_workers=4) as client:
    generate_sliver("dia_object_collection")
    generate_sliver("object_photoz")

In [ ]:
with Client(n_workers=1) as client:
    generate_sliver("object_collection")

The nested default columns were not propagated properly, so I had to set them manually. I'll investigate and file an issue if needed.